# [실습 프로젝트] Naive RAG 구현 

- 각 단계별 지시사항에 따라 코드를 완성하세요. 
- 제시된 지시사항과 LangChain 문서를 참조하여 시스템을 구성합니다. 

`(1) 벡터 저장소 설정`
- HuggingFace에서 지원하는 BAAI/bge-m3 임베딩 모델을 사용하여 문서를 벡터화
- FAISS DB를 벡터 스토어로 사용 (IndexFlatL2 사용: 유클리드 거리)

In [1]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings  

# Hugging Face의 임베딩 모델 생성
embeddings_model = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

# 임베딩 차원 확인
embedding = embeddings_model.embed_query("test")
print(f"임베딩 차원: {len(embedding)}")

임베딩 차원: 1024


In [ ]:
# Ollama 임베딩 모델을 사용한 FAISS 벡터 저장소 생성
import faiss 
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

# FAISS 인덱스 초기화 (유클리드 거리 사용)
dim = len(embedding)  # 임베딩 차원
faiss_index = faiss.IndexFlatL2(dim)

# FAISS 벡터 저장소 생성
faiss_db = FAISS(
    embedding_function=embeddings_model,
    index=faiss_index,           # 벡터 검색을 위한 데이터 구조를 정의
    docstore=InMemoryDocstore(), # 문서 저장소 객체를 지정 - 문서의 원본 내용과 메타데이터를 보관
    index_to_docstore_id={},     # 인덱스와 문서 간의 연결을 관리 (매핑 딕셔너리)
)

# 저장된 문서의 갯수 확인
print(faiss_db.index.ntotal)

0


In [ ]:
import uuid
from langchain_core.documents import Document

documents = [
    ("인공지능은 컴퓨터 과학의 한 분야입니다.", "AI 개론"),
    ("머신러닝은 인공지능의 하위 분야입니다.", "AI 개론"),
    ("딥러닝은 머신러닝의 한 종류입니다.", "딥러닝 입문"),
    ("자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.", "AI 개론"),
    ("컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.", "딥러닝 입문")
]

doc_objects = []
for content, source in documents:
    doc = Document(
        page_content=content,
        metadata={"source": source},
    )
    doc_objects.append(doc)

# 문서 id 생성
doc_ids = [str(uuid.uuid4()) for _ in range(len(doc_objects))]

# 문서를 벡터 저장소에 저장
added_doc_ids = faiss_db.add_documents(documents=doc_objects, ids=doc_ids)

# 벡터 저장소에 저장된 문서를 확인
print(f"{len(added_doc_ids)}개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.")
print(f"문서 IDs: {added_doc_ids}")
print(f"저장된 총 문서 개수: {faiss_db.index.ntotal}")

5개의 문서가 성공적으로 벡터 저장소에 추가되었습니다.
['5a454e3e-5b83-4377-a52e-3da0f4e24eb9', 'cc143568-a9f8-41bc-ab14-b6f84f71d75b', '4d1625ec-8c7b-44d4-afce-3254c3b29328', '588771d1-b346-426c-95f0-623320f22ff6', '639d27fc-2481-450d-822f-00f3aa383651']


`(2) 검색기 정의`
- mmr 검색으로 상위 3개 문서 검색하는 Retriever 사용
- 다양성을 높이는 설정을 사용 

In [ ]:
# mmr 검색기 생성
faiss_mmr_retriever = faiss_db.as_retriever(
    search_type="mmr",      # Maximum Marginal Relevance 검색 사용
    search_kwargs={
        "k": 3,             # 상위 3개 문서 검색
        "lambda_mult": 0.9, # 다양성 매개변수 (0에 가까울수록 다양성 높음, 1에 가까울수록 관련성 높음)
        "fetch_k": 10       # MMR에서 고려할 문서 개수 (더 많은 후보에서 선택)
    }
)

In [ ]:
# 검색 테스트 
query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
retrieved_docs = faiss_mmr_retriever.invoke(query)

print(f"쿼리: {query}")
print("검색 결과:")
print("=" * 80)
for i, doc in enumerate(retrieved_docs, 1):
    content = doc.page_content
    source = doc.metadata.get("source", "Unknown")
    
    # 내용이 100자 이하면 전체 출력, 초과하면 앞뒤 100자씩 출력
    if len(content) <= 100:
        display_content = content
    else:
        display_content = f"{content[:50]}...{content[-50:]}"
    
    print(f"[문서 {i}] 출처: {source}")
    print(f"내용: {display_content}")
    print("-" * 80)

쿼리: 대표적인 시퀀스 모델은 어떤 것들이 있나요?
검색 결과:
[문서 1] 출처: 딥러닝 입문
내용: 딥러닝은 머신러닝의 한 종류입니다.
--------------------------------------------------------------------------------
[문서 2] 출처: 딥러닝 입문
내용: 컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.
--------------------------------------------------------------------------------
[문서 3] 출처: AI 개론
내용: 인공지능은 컴퓨터 과학의 한 분야입니다.
--------------------------------------------------------------------------------


`(3) RAG 프롬프트 구성`

- 작성 기준: 
    - LangChain의 ChatPromptTemplate 클래스 사용
    - 변수 처리는 {context}, {question} 형식 사용
    - 답변은 한글로 출력되도록 프롬프트 작성
    
- 아래 템플릿 코드를 기반으로 다음 내용을 참고하여 작성합니다. 

    1. 프롬프트 구성요소:
        - 작업 지침
        - 컨텍스트 영역
        - 질문 영역
        - 답변 형식 가이드

    2. 작업 지침:
        - 컨텍스트 기반 답변 원칙
        - 외부 지식 사용 제한
        - 불확실성 처리 방법
        - 답변 불가능한 경우의 처리 방법

    3. 답변 형식:
        - 핵심 답변 섹션
        - 근거 제시 섹션
        - 추가 설명 섹션 (필요시)

    4. 제약사항 반영:
        - 답변은 사실에 기반해야 함
        - 추측이나 가정을 최소화해야 함
        - 명확한 근거 제시가 필요함
        - 구조화된 형태로 작성되어야 함

In [ ]:
# Prompt 템플릿 (예시)
from langchain.prompts import ChatPromptTemplate

template = """Answer the question based only on the following context.

[Context]
{context}

[Question] 
{question}

[Answer]
"""

prompt = ChatPromptTemplate.from_template(template)

In [ ]:
# Prompt 템플릿 (여기에 작성하세요)
from langchain.prompts import ChatPromptTemplate

template = """당신은 주어진 문서 컨텍스트를 기반으로 정확하고 유용한 답변을 제공하는 AI 어시스턴트입니다.

## 작업 지침
1. **컨텍스트 기반 답변 원칙**: 반드시 제공된 컨텍스트 정보만을 사용하여 답변하세요.
2. **외부 지식 사용 제한**: 컨텍스트에 없는 정보나 개인적인 지식을 추가하지 마세요.
3. **불확실성 처리**: 컨텍스트 정보가 불충분하거나 모호한 경우, 이를 명시하세요.
4. **답변 불가능한 경우**: 컨텍스트에서 답변할 수 없는 질문이라면, 정직하게 답변할 수 없다고 말하세요.

## 컨텍스트
{context}

## 질문
{question}

## 답변 형식
다음 구조에 따라 한글로 답변하세요:

**[핵심 답변]**
질문에 대한 직접적이고 명확한 답변을 제시하세요.

**[근거 제시]**
답변의 근거가 되는 컨텍스트 정보를 명시하세요.

**[추가 설명]** (필요시)
답변을 보완하는 관련 정보나 맥락을 제공하세요.

## 제약사항
- 답변은 반드시 사실에 기반해야 합니다
- 추측이나 가정을 최소화하세요
- 컨텍스트에서 명확한 근거를 찾을 수 있는 경우에만 답변하세요
- 구조화된 형태로 명확하게 작성하세요

답변:"""

prompt = ChatPromptTemplate.from_template(template)

# 템플릿 출력
prompt.pretty_print()

test_context = "\n".join([doc.page_content for doc in retrieved_docs])
test_question = "딥러닝과 머신러닝의 관계는 무엇인가요?"

formatted_prompt = prompt.format(
    context=test_context,
    question=test_question
)

print("포맷팅된 프롬프트:")
print("=" * 80)
print(formatted_prompt)
print("=" * 80)

================================ Human Message =================================

당신은 주어진 문서 컨텍스트를 기반으로 정확하고 유용한 답변을 제공하는 AI 어시스턴트입니다.

## 작업 지침
1. **컨텍스트 기반 답변 원칙**: 반드시 제공된 컨텍스트 정보만을 사용하여 답변하세요.
2. **외부 지식 사용 제한**: 컨텍스트에 없는 정보나 개인적인 지식을 추가하지 마세요.
3. **불확실성 처리**: 컨텍스트 정보가 불충분하거나 모호한 경우, 이를 명시하세요.
4. **답변 불가능한 경우**: 컨텍스트에서 답변할 수 없는 질문이라면, 정직하게 답변할 수 없다고 말하세요.

## 컨텍스트
{context}

## 질문
{question}

## 답변 형식
다음 구조에 따라 한글로 답변하세요:

**[핵심 답변]**
질문에 대한 직접적이고 명확한 답변을 제시하세요.

**[근거 제시]**
답변의 근거가 되는 컨텍스트 정보를 명시하세요.

**[추가 설명]** (필요시)
답변을 보완하는 관련 정보나 맥락을 제공하세요.

## 제약사항
- 답변은 반드시 사실에 기반해야 합니다
- 추측이나 가정을 최소화하세요
- 컨텍스트에서 명확한 근거를 찾을 수 있는 경우에만 답변하세요
- 구조화된 형태로 명확하게 작성하세요

답변:
포맷팅된 프롬프트:
Human: 당신은 주어진 문서 컨텍스트를 기반으로 정확하고 유용한 답변을 제공하는 AI 어시스턴트입니다.

## 작업 지침
1. **컨텍스트 기반 답변 원칙**: 반드시 제공된 컨텍스트 정보만을 사용하여 답변하세요.
2. **외부 지식 사용 제한**: 컨텍스트에 없는 정보나 개인적인 지식을 추가하지 마세요.
3. **불확실성 처리**: 컨텍스트 정보가 불충분하거나 모호한 경우, 이를 명시하세요.
4. **답변 불가능한 경우**: 컨텍스트에서 답변할 수 없는 질문이라면, 정직하게 답변할 수 없다고 말하세요.

## 컨텍스트
딥러닝은 머신러닝의 한 종류입니다.
컴퓨터

`(4) RAG 체인 구성`
- LangChain의 LCEL 문법을 사용
- 검색 결과를 프롬프트의 'context'로 전달하고,
- 사용자가 입력한 질문을 그래도 프롬프트의 'question'에 전달
- LLM 설정:
    - ChatOpenAI 사용 ('gpt-4.1-mini' 모델)
    - temperature: 답변의 일관성을 가져가는 설정값을 사용 
    - 기타 필요한 설정 
- 출력 파서: 문자열 부분만 출력되도록 구성 

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

# LLM 설정
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.7,
    top_p=0.9,           
    frequency_penalty=0.3,
    presence_penalty=0.3
)


# 문서 포맷팅
def format_docs(docs):
    """검색된 문서들을 문자열로 포맷팅"""
    formatted_docs = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get("source", "Unknown")
        content = doc.page_content
        formatted_docs.append(f"[문서 {i}] 출처: {source}\n내용: {content}")
    return "\n\n".join(formatted_docs)

# RAG 체인 생성
rag_chain = (
    {
        "context": faiss_mmr_retriever | format_docs,  # 검색 결과를 포맷팅하여 context로 전달
        "question": RunnablePassthrough()              # 사용자 질문을 그대로 question으로 전달
    }
    | prompt        # 프롬프트 템플릿에 context와 question 삽입
    | llm          # LLM으로 응답 생성
    | StrOutputParser()  # 문자열 출력 파서로 결과 정리
)

# 체인 실행
query = "대표적인 시퀀스 모델은 어떤 것들이 있나요?"
print(f"쿼리: {query}")
print("=" * 80)

try:
    output = rag_chain.invoke(query)
    print("답변:")
    print(output)
except Exception as e:
    print(f"오류 발생: {e}")
    print("OpenAI API 키가 설정되지 않았거나 모델에 접근할 수 없습니다.")
    print("실제 환경에서는 OpenAI API 키를 설정해야 합니다.")

print("=" * 80)


쿼리: 대표적인 시퀀스 모델은 어떤 것들이 있나요?
답변:
**[핵심 답변]**  
대표적인 시퀀스 모델에 대한 정보는 제공된 컨텍스트에 포함되어 있지 않습니다.

**[근거 제시]**  
컨텍스트에는 딥러닝, 컴퓨터 비전, 인공지능에 대한 정보만 포함되어 있으며, 시퀀스 모델에 대한 언급은 없습니다.

**[추가 설명]**  
따라서, 시퀀스 모델에 대한 구체적인 답변을 제공할 수 없습니다.


`(5) Gradio 스트리밍 구현`
- ChatInterface 사용
- `chain.stream()`으로 응답을 청크 단위로 스트리밍

In [2]:
import gradio as gr
from typing import Iterator

# 스트리밍 응답 생성 함수
def get_streaming_response(message: str, history) -> Iterator[str]:
    """
    RAG 체인을 사용하여 스트리밍 응답을 생성하는 함수
    
    Args:
        message (str): 사용자 입력 메시지
        history: 채팅 히스토리 (ChatInterface에서 자동으로 관리)
    
    Yields:
        str: 누적된 응답 텍스트
    """

    # RAG Chain 실행 및 스트리밍 응답 생성
    response = ""
    
    # rag_chain.stream()을 사용하여 청크 단위로 응답 받기
    for chunk in rag_chain.stream(message):
        if isinstance(chunk, str):
            response += chunk
            yield response
        elif hasattr(chunk, 'content'):  # AIMessage 객체인 경우
            response += chunk.content
            yield response


# Gradio 인터페이스 설정
demo = gr.ChatInterface(
    fn=get_streaming_response,
    title="🤖 RAG 기반 AI 챗봇",
    description="...",
    examples=[
        "인공지능이란 무엇인가요?",
        "머신러닝과 딥러닝의 차이점은 무엇인가요?",
        "자연어 처리 기술에 대해 설명해주세요.",
        "컴퓨터 비전은 어떤 분야인가요?",
        "딥러닝이 머신러닝의 하위 분야인 이유는?"
    ],
)

# 실행
if __name__ == "__main__":
    demo.launch()

e:\study\modulab-ai\week1\faq_bot\.venv\Lib\site-packages\gradio\chat_interface.py:345: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [3]:
# demo 실행 종료
demo.close()

Closing server running on port: 7860
